# Experiment 1 - per-configuration diagnostic figures

Writes one folder per config under `experiment_figs/`:
- `w_comparison_cell_*.png` - estimated vs model w (Hovmoller + time series), built from saved arrays (fast, no model needed).
- `velocity_map.png` - depth/time-mean U, V, W with the array overlaid (loads the model once).

Idempotent: any figure that already exists on disk is skipped, so re-running only
fills gaps. The model is loaded only if at least one config is missing its velocity_map.

Run in place after `run_experiment.py`.

In [1]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

REPO = "/home/edavenport/analysis/tpose24-osse"
HERE = os.path.join(REPO, "experiment_1")
sys.path.insert(0, REPO)
import osse_tools as ot

DATA = os.path.join(HERE, "data")
EFIG = os.path.join(HERE, "experiment_figs"); os.makedirs(EFIG, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
m = pd.read_csv(os.path.join(DATA, "metrics.csv"))
print(f"{len(m)} cells / {m.config.nunique()} configs")

96 cells / 66 configs


## Per-cell w comparison (from saved arrays)

In [2]:
# Per-cell w comparison (Hovmoller + time series) from the saved arrays - no model load.
# Skip any figure already on disk so completed configs are not redone.
n_new = 0
for _, r in m.iterrows():
    outdir  = os.path.join(EFIG, r.config)
    outpath = os.path.join(outdir, f"w_comparison_cell_{r.center_lat:+.2f}.png")
    if os.path.exists(outpath):
        continue
    ds = xr.open_dataset(os.path.join(HERE, r.nc_path))
    fig = ot.plot_w_comparison(ds.w_est, ds.w_model, point_depth=-50)
    fig.suptitle(f"{r.config}  |  cell {r.center_lat:+.1f}N  |  "
                 f"RMS/sigma={r.norm_rms:.2f}  r={r['corr']:.2f}", y=1.01, fontsize=12)
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(outpath, dpi=130, bbox_inches="tight")
    plt.close(fig)
    n_new += 1
print(f"wrote {n_new} new per-cell w_comparison figures (skipped existing)")

wrote 0 new per-cell w_comparison figures (skipped existing)


## Per-config velocity maps (loads the model)

In [3]:
# Per-config velocity/vorticity context maps. These need the model, so load it
# only if at least one config is still missing its velocity_map.png.
cfg_paths = sorted(glob.glob(os.path.join(HERE, "configs", "**", "*.json"), recursive=True))
pending = [p for p in cfg_paths
           if not os.path.exists(
               os.path.join(EFIG, json.load(open(p))["name"], "velocity_map.png"))]
print(f"{len(pending)} configs need a velocity_map "
      f"({len(cfg_paths) - len(pending)} already have one)")

if pending:
    RUN_DIR = "/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons"
    ITERS   = list(range(36, 26173, 36))
    ds_model = ot.load_model(RUN_DIR, ITERS).sel(time=slice("2012-10-11", None))

    for path in pending:
        cfg = json.load(open(path))
        cells = ot.load_cells(path)
        positions = sorted({p for _, pos in cells for p in pos})
        cells_plot = [(f"{cl:+.1f}", pos, f"C{i}") for i, (cl, pos) in enumerate(cells)]
        fig = ot.plot_velocity_map(ds_model, positions, max_depth=70, cells=cells_plot)
        fig.suptitle(f"{cfg['name']}  ({cfg['description']})", fontsize=11, y=1.02)
        outdir = os.path.join(EFIG, cfg["name"]); os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, "velocity_map.png"), dpi=130, bbox_inches="tight")
        plt.close(fig)
    print(f"wrote {len(pending)} per-config velocity maps")
else:
    print("all velocity maps present - skipped model load")

21 configs need a velocity_map (45 already have one)


/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

...

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/xmitgcm/mds_store.py:913: UserWarning: Couldn't find available_diagnostics.log in /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons or /data/SO3/edavenport/tpose24/oct2012_3month_transp_cons. Using default version.
  warnings.warn

wrote 21 per-config velocity maps
